# CRW DHW Matchup Pipeline

Extracts NOAA Coral Reef Watch (CRW) Degree Heating Week (DHW) values for each coral survey record by spatially and temporally matching survey coordinates/dates to daily global DHW NetCDF files.

**Inputs:**
- `merged_bleaching_data.csv` — coral survey dataset with `Date`, `Latitude_Degrees`, `Longitude_Degrees`
- CRW daily DHW NetCDF files (`ct5km_dhw_v3.1_YYYYMMDD.nc`)

**Outputs:**
- Per-year CSVs with appended `DHW_CRW` column
- Combined CSV spanning all years

## Configuration

Set all paths here before running.

In [ ]:
# ----- Paths -----
SURVEY_PATH  = "data/merged_bleaching_data.csv"   # coral survey CSV
DHW_FOLDER   = "data/global_daily_dhw"            # folder of CRW .nc files
OUTPUT_FOLDER = "data/with_dhw"                   # where per-year outputs are saved

# ----- Parameters -----
YEARS = [str(y) for y in range(1985, 2021)]       # years to process
N_JOBS = -1                                        # parallel workers (-1 = all cores)

## Imports

In [ ]:
import os
import glob
import warnings

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from joblib import Parallel, delayed
from tqdm import tqdm

warnings.filterwarnings("ignore")
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

## Load Survey Data

In [ ]:
df_survey = pd.read_csv(SURVEY_PATH, parse_dates=["Date"])
df_survey["date_str"] = df_survey["Date"].dt.strftime("%Y%m%d")
print(f"Survey records: {len(df_survey):,}  |  Date range: {df_survey['Date'].min().date()} – {df_survey['Date'].max().date()}")

## Survey Site Map

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), subplot_kw={"projection": ccrs.PlateCarree()})

ax.scatter(
    df_survey["Longitude_Degrees"],
    df_survey["Latitude_Degrees"],
    s=6, alpha=0.4, color="steelblue", label=f"Survey sites (n={len(df_survey):,})"
)

ax.coastlines(linewidth=0.6)
ax.add_feature(cfeature.LAND, facecolor="#e8e8e8")
ax.add_feature(cfeature.BORDERS, linestyle=":", linewidth=0.4)
ax.set_extent([-180, 180, -60, 60], crs=ccrs.PlateCarree())

gl = ax.gridlines(draw_labels=True, linestyle="--", linewidth=0.4, alpha=0.6)
gl.top_labels = False
gl.right_labels = False

ax.set_title("Global Coral Survey Sites", fontsize=13)
ax.legend(loc="lower left", fontsize=9)
plt.tight_layout()
plt.show()

## Index CRW NetCDF Files

Builds a lookup dictionary mapping date strings (`YYYYMMDD`) to file paths.

In [ ]:
nc_index = {
    os.path.basename(f).split("_")[-1].split(".")[0]: f
    for f in glob.glob(os.path.join(DHW_FOLDER, "*.nc"))
}
print(f"Indexed {len(nc_index):,} CRW DHW files.")

## DHW Extraction — Per-Year Matchup

For each survey record, opens the corresponding daily CRW NetCDF file and extracts the nearest-neighbor DHW value. Runs in parallel across all CPU cores.

In [ ]:
def extract_dhw(row):
    """Extract DHW value for a single survey record."""
    date_str = row["date_str"]
    if date_str not in nc_index:
        return row.name, np.nan
    try:
        with xr.open_dataset(nc_index[date_str]) as ds:
            dhw = ds["degree_heating_week"].squeeze()
            mask = ds["mask"].squeeze()
            dhw = dhw.where(mask == 0)
            value = dhw.sel(
                lat=row["Latitude_Degrees"],
                lon=row["Longitude_Degrees"],
                method="nearest"
            ).values.item()
            return row.name, value
    except Exception:
        return row.name, np.nan


for year in YEARS:
    df_year = df_survey[df_survey["Date"].dt.year == int(year)].copy()
    if df_year.empty:
        continue

    results = Parallel(n_jobs=N_JOBS)(
        delayed(extract_dhw)(row)
        for _, row in tqdm(df_year.iterrows(), total=len(df_year), desc=year)
    )

    for idx, dhw_val in results:
        df_year.at[idx, "DHW_CRW"] = dhw_val

    out_path = os.path.join(OUTPUT_FOLDER, f"bleaching_{year}_with_dhw.csv")
    df_year.drop(columns=["date_str"]).to_csv(out_path, index=False)

print("Extraction complete.")

## Combine Yearly Outputs

In [ ]:
csv_files = sorted(glob.glob(os.path.join(OUTPUT_FOLDER, "bleaching_*_with_dhw.csv")))

df_combined = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)

combined_path = os.path.join(OUTPUT_FOLDER, "bleaching_1985-2020_with_dhw_combined.csv")
df_combined.to_csv(combined_path, index=False)

print(f"Combined {len(csv_files)} files → {len(df_combined):,} records")
print(f"DHW coverage: {df_combined['DHW_CRW'].notna().mean():.1%} of records matched")
df_combined.head()